# Line-Art Illustration Generation with Stable Diffusion

Generate clean, engaging line-art style cooking illustrations using Stable Diffusion with ControlNet.

**Purpose**: Create visual guides for each cooking step

**Model**: Stable Diffusion v1.5 + ControlNet (Canny/Lineart)

## Setup and Imports

In [ ]:
# Imports
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from diffusers import UniPCMultistepScheduler
from PIL import Image, ImageDraw, ImageFont
import numpy as np
from controlnet_aux import LineartDetector
import json
from pathlib import Path
import uuid

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("✅ Imports complete")

## Load ControlNet Model

Using ControlNet with lineart conditioning for clean line-style illustrations.

In [ ]:
# Load ControlNet for lineart
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_lineart",
    torch_dtype=torch.float16
)

# Load Stable Diffusion pipeline
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None  # Optional: disable for cooking images
)

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipe.to(device)

# Speed optimization
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()  # Save memory

print(f"✅ Model loaded on {device}")

## Method 1: Text-to-Image Lineart Generation

In [ ]:
def generate_cooking_illustration(cooking_step: dict, output_path: str):
    """
    Generate line-art illustration from cooking step description.
    
    Args:
        cooking_step: Dict with 'instruction_text' and 'cooking_method'
        output_path: Where to save the illustration
    """
    
    # Extract cooking action
    instruction = cooking_step['instruction_text']
    method = cooking_step['cooking_method']
    
    # Craft prompt for line-art style
    prompt = f"""
    simple line art drawing, black and white, clean lines, minimalist style,
    cooking instruction: {method}, {instruction},
    single object, white background, technical illustration,
    no shading, outline only, beginner-friendly visual guide
    """.strip()
    
    # Negative prompt to avoid unwanted elements
    negative_prompt = """
    photo, photograph, realistic, colored, shading, gradient,
    complex, cluttered, text, watermark, multiple views
    """.strip()
    
    # Generate image
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=20,  # Faster: 20, Better quality: 50
        guidance_scale=7.5,
        height=512,
        width=512,
        generator=torch.Generator(device=device).manual_seed(42)
    ).images[0]
    
    # Post-process: Convert to pure line art
    image = image.convert('L')  # Grayscale
    image = image.point(lambda x: 0 if x < 128 else 255, '1')  # Binary
    
    # Save
    image.save(output_path)
    
    return image

# Test example
test_step = {
    "instruction_text": "Dice the chicken breast into 1-inch cubes",
    "cooking_method": "dice"
}

output_path = "data/results/illustrations/test_dice_chicken.png"
Path(output_path).parent.mkdir(parents=True, exist_ok=True)

illustration = generate_cooking_illustration(test_step, output_path)
print(f"✅ Illustration saved to {output_path}")
illustration  # Display in notebook

## Method 2: Using Pre-made Sketch as ControlNet Input

For more control over the output, you can provide a simple sketch.

In [ ]:
def create_simple_sketch(cooking_method: str, size=(512, 512)):
    """
    Create a simple geometric sketch as ControlNet input.
    This gives more control over composition.
    """
    img = Image.new('RGB', size, color='white')
    draw = ImageDraw.Draw(img)
    
    # Simple sketches for common actions
    if cooking_method == 'chop' or cooking_method == 'dice':
        # Draw knife and cutting board
        draw.rectangle([100, 300, 400, 320], outline='black', width=3)  # Cutting board
        draw.line([200, 150, 250, 300], fill='black', width=3)  # Knife
        draw.ellipse([230, 280, 280, 320], outline='black', width=2)  # Food item
    
    elif cooking_method == 'sauté' or cooking_method == 'fry':
        # Draw pan on stove
        draw.ellipse([150, 200, 350, 250], outline='black', width=3)  # Pan
        draw.line([150, 225, 120, 225], fill='black', width=3)  # Handle
        draw.rectangle([200, 350, 300, 380], outline='black', width=2)  # Stove
    
    elif cooking_method == 'mix' or cooking_method == 'stir':
        # Draw bowl and spoon
        draw.arc([150, 200, 350, 400], 0, 180, fill='black', width=3)  # Bowl
        draw.line([256, 150, 256, 300], fill='black', width=3)  # Spoon
    
    else:
        # Generic: centered rectangle
        draw.rectangle([180, 180, 330, 330], outline='black', width=3)
    
    return img

def generate_with_sketch_control(cooking_step: dict, output_path: str):
    """
    Generate lineart using a sketch as ControlNet guidance.
    """
    method = cooking_step['cooking_method']
    instruction = cooking_step['instruction_text']
    
    # Create control sketch
    control_image = create_simple_sketch(method)
    
    # Convert to lineart format for ControlNet
    processor = LineartDetector.from_pretrained("lllyasviel/Annotators")
    control_image = processor(control_image)
    
    # Prompt
    prompt = f"""
    clean line art illustration, simple black lines on white background,
    cooking step: {instruction}, minimalist kitchen diagram,
    instructional drawing, beginner-friendly
    """.strip()
    
    # Generate with ControlNet
    image = pipe(
        prompt=prompt,
        image=control_image,
        num_inference_steps=20,
        controlnet_conditioning_scale=0.8,  # How strictly to follow sketch
        guidance_scale=7.5,
    ).images[0]
    
    # Save
    image.save(output_path)
    return image

# Test
test_step = {
    "instruction_text": "Sauté the vegetables until golden brown",
    "cooking_method": "sauté"
}

output = "data/results/illustrations/test_saute_controlled.png"
img = generate_with_sketch_control(test_step, output)
print(f"✅ Controlled illustration saved")
img

## Method 3: Batch Generation for All Recipe Steps

In [ ]:
def generate_recipe_illustrations(recipe: dict, output_dir: str):
    """
    Generate illustrations for all cooking steps in a recipe.
    
    Args:
        recipe: Recipe dict with 'cooking_steps' array
        output_dir: Directory to save illustrations
    """
    from tqdm import tqdm
    import time
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    illustrations = []
    
    for step in tqdm(recipe['cooking_steps'], desc="Generating illustrations"):
        step_num = step['step_number']
        file_path = output_path / f"step_{step_num:02d}.png"
        
        # Generate illustration
        start_time = time.time()
        img = generate_cooking_illustration(step, str(file_path))
        gen_time = (time.time() - start_time) * 1000  # ms
        
        # Create illustration metadata
        illustration = {
            "illustration_id": str(uuid.uuid4()),
            "step_id": step.get('step_id', f"step_{step_num}"),
            "image_data": {
                "format": "PNG",
                "file_path": str(file_path),
                "width_px": 512,
                "height_px": 512
            },
            "style_metadata": {
                "line_weight": 2.0,
                "color_scheme": "monochrome",
                "complexity": "simple"
            },
            "generation_metadata": {
                "model": "stable-diffusion-v1.5 + controlnet-lineart",
                "generation_time_ms": gen_time,
                "quality_score": 0.85  # Placeholder
            },
            "alt_text": f"Illustration showing {step['cooking_method']}: {step['instruction_text'][:100]}"
        }
        
        illustrations.append(illustration)
        
        # Add to step
        step['illustration'] = illustration
    
    return illustrations

# Test with sample recipe
sample_recipe = {
    "recipe_id": "test-recipe-001",
    "cooking_steps": [
        {
            "step_number": 1,
            "instruction_text": "Dice the chicken breast into 1-inch cubes",
            "cooking_method": "dice"
        },
        {
            "step_number": 2,
            "instruction_text": "Heat oil in a large pan over medium-high heat",
            "cooking_method": "heat"
        },
        {
            "step_number": 3,
            "instruction_text": "Sauté the chicken until golden brown, about 5-7 minutes",
            "cooking_method": "sauté"
        }
    ]
}

illustrations = generate_recipe_illustrations(
    sample_recipe,
    "data/results/illustrations/test-recipe-001"
)

print(f"✅ Generated {len(illustrations)} illustrations")
print(f"Average generation time: {np.mean([i['generation_metadata']['generation_time_ms'] for i in illustrations]):.0f}ms")

## Quality Check and Validation

In [ ]:
def validate_illustration_quality(image_path: str) -> dict:
    """
    Automated quality check for generated illustrations.
    
    Returns:
        Dict with quality metrics
    """
    import cv2
    
    # Load image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    
    # Check 1: Not blank (pixel variance > threshold)
    variance = np.var(img)
    is_not_blank = variance > 100
    
    # Check 2: Has clear lines (edge detection)
    edges = cv2.Canny(img, 50, 150)
    edge_density = np.sum(edges > 0) / edges.size
    has_clear_lines = 0.01 < edge_density < 0.3
    
    # Check 3: Good contrast
    contrast = img.std()
    good_contrast = contrast > 50
    
    # Calculate quality score (0-1)
    quality_score = (
        (0.4 if is_not_blank else 0) +
        (0.4 if has_clear_lines else 0) +
        (0.2 if good_contrast else 0)
    )
    
    return {
        "quality_score": quality_score,
        "passed": quality_score >= 0.6,
        "checks": {
            "not_blank": is_not_blank,
            "clear_lines": has_clear_lines,
            "good_contrast": good_contrast
        },
        "metrics": {
            "variance": float(variance),
            "edge_density": float(edge_density),
            "contrast": float(contrast)
        }
    }

# Test validation
test_image = "data/results/illustrations/test_dice_chicken.png"
quality = validate_illustration_quality(test_image)

print(f"Quality Score: {quality['quality_score']:.2f}")
print(f"Passed: {'✅' if quality['passed'] else '❌'}")
print(f"Checks: {quality['checks']}")

## Save Illustration Metadata

In [ ]:
# Save all illustrations metadata
metadata_path = "data/results/illustrations/test-recipe-001/metadata.json"

with open(metadata_path, 'w') as f:
    json.dump({
        "recipe_id": sample_recipe['recipe_id'],
        "illustrations": illustrations,
        "total_count": len(illustrations),
        "generation_summary": {
            "model": "stable-diffusion-v1.5 + controlnet-lineart",
            "avg_time_ms": np.mean([i['generation_metadata']['generation_time_ms'] for i in illustrations]),
            "total_time_ms": sum([i['generation_metadata']['generation_time_ms'] for i in illustrations])
        }
    }, f, indent=2)

print(f"✅ Metadata saved to {metadata_path}")

## Summary

This notebook demonstrates line-art illustration generation using:
- Stable Diffusion v1.5 + ControlNet
- Text-to-image generation with lineart prompts
- Sketch-guided generation for more control
- Automated quality validation

**Performance**: ~3-5 seconds per illustration on GPU, ~20-30 seconds on CPU

**Next Steps**:
1. Fine-tune prompts for better consistency
2. Create template library for common actions
3. Implement caching to avoid regeneration
4. Add SVG conversion for scalability